In [1]:
#Logging setup
import logging, sys

logger = logging.getLogger("api.middleware")
logger.setLevel(logging.DEBUG)
logger.propagate = False          # notebook ke root logger se double output na ho

if not logger.handlers:
    fmt = logging.Formatter("%(asctime)s | %(levelname)-8s | %(message)s", "%H:%M:%S")
    console = logging.StreamHandler(sys.stdout)
    console.setFormatter(fmt)
    logfile = logging.FileHandler("middleware.log", mode="w", encoding="utf-8")
    logfile.setFormatter(fmt)
    logger.addHandler(console)
    logger.addHandler(logfile)

logger.info("Logging ready")

11:42:47 | INFO     | Logging ready


In [2]:
#Middleware (3 decorators)
import time
from functools import wraps

SENSITIVE_KEYS = {"password", "token", "secret", "api_key"}
SLOW_MS = 100


class PermissionDenied(Exception):
    """Role ko access ki ijazat nahi."""


def _mask(value):
    """Dict ke sensitive fields chupao (nested dicts samet)."""
    if isinstance(value, dict):
        return {k: "***" if str(k).lower() in SENSITIVE_KEYS else _mask(v)
                for k, v in value.items()}
    return value


def logged(func):
    """Input, output aur errors log karo."""
    @wraps(func)
    def wrapper(*args, **kwargs):
        name = func.__qualname__
        logger.info("CALL   %s | args=%s kwargs=%s",
                    name, tuple(_mask(a) for a in args), _mask(kwargs))
        try:
            result = func(*args, **kwargs)
        except Exception as e:
            logger.error("FAIL   %s | %s: %s", name, type(e).__name__, e)
            raise
        logger.info("RETURN %s | %r", name, _mask(result))
        return result
    return wrapper


def timed(func):
    """Execution time naapo (error aaye tab bhi)."""
    @wraps(func)
    def wrapper(*args, **kwargs):
        start = time.perf_counter()
        try:
            return func(*args, **kwargs)
        finally:
            ms = (time.perf_counter() - start) * 1000
            level = logging.WARNING if ms > SLOW_MS else logging.INFO
            logger.log(level, "TIMER  %s | %.2f ms%s",
                       func.__qualname__, ms, " (SLOW)" if ms > SLOW_MS else "")
    return wrapper


def require_role(*allowed_roles):
    """Sirf allowed roles ko chalne do. Ye decorator factory hai (arguments leta hai)."""
    def decorator(func):
        @wraps(func)
        def wrapper(*args, **kwargs):
            user = kwargs.get("user") or (args[0] if args else None)
            role = user.get("role") if isinstance(user, dict) else None
            who = user.get("name", "?") if isinstance(user, dict) else "?"

            if role not in allowed_roles:
                logger.warning("DENIED %s | user=%s role=%s needs=%s",
                               func.__qualname__, who, role, list(allowed_roles))
                raise PermissionDenied(
                    f"{who} ({role}) ko {func.__qualname__} ki ijazat nahi")
            logger.debug("ALLOWED %s | user=%s role=%s", func.__qualname__, who, role)
            return func(*args, **kwargs)
        return wrapper
    return decorator

In [3]:
#Original functions + wrapping
# ---------- Original functions (koi decorator nahi) ----------
def get_orders(user, limit=3):
    return [{"id": i, "total": 100 * i} for i in range(1, limit + 1)]

def update_profile(user, data):
    return {"status": "updated", "user": user["name"], "data": data}

def delete_user(user, target_id):
    time.sleep(0.15)                       # slow operation ka demo
    if target_id == 0:
        raise ValueError("target_id 0 invalid hai")
    return {"deleted": target_id}


# ---------- Wrapping (decorators stack karna) ----------
api_get_orders = logged(require_role("admin", "editor", "viewer")(timed(get_orders)))
api_update_profile = logged(require_role("admin", "editor")(timed(update_profile)))
api_delete_user = logged(require_role("admin")(timed(delete_user)))

In [4]:
#Test
admin  = {"name": "Malika", "role": "admin"}
editor = {"name": "Ahmed",  "role": "editor"}
guest  = {"name": "Guest",  "role": "viewer"}

def attempt(label, fn, *args, **kwargs):
    logger.info("---- %s ----", label)
    try:
        fn(*args, **kwargs)
    except PermissionDenied:
        pass                               # already WARNING log ho chuka
    except Exception:
        pass                               # already ERROR log ho chuka

attempt("viewer orders dekhta hai",   api_get_orders, guest, limit=2)
attempt("editor profile update",      api_update_profile, editor,
        {"city": "Lahore", "password": "my-secret-123"})
attempt("viewer profile update",      api_update_profile, guest, {"city": "X"})
attempt("admin delete (slow)",        api_delete_user, admin, 42)
attempt("editor delete (denied)",     api_delete_user, editor, 42)
attempt("admin delete (error)",       api_delete_user, admin, 0)

# Original function wrapper ke andar safe hai:
logger.info("Original intact: %s | wrapped: %s",
            api_delete_user.__name__, api_delete_user.__wrapped__.__name__)

11:44:30 | INFO     | ---- viewer orders dekhta hai ----
11:44:30 | INFO     | CALL   get_orders | args=({'name': 'Guest', 'role': 'viewer'},) kwargs={'limit': 2}
11:44:30 | DEBUG    | ALLOWED get_orders | user=Guest role=viewer
11:44:30 | INFO     | TIMER  get_orders | 0.01 ms
11:44:30 | INFO     | RETURN get_orders | [{'id': 1, 'total': 100}, {'id': 2, 'total': 200}]
11:44:30 | INFO     | ---- editor profile update ----
11:44:30 | INFO     | CALL   update_profile | args=({'name': 'Ahmed', 'role': 'editor'}, {'city': 'Lahore', 'password': '***'}) kwargs={}
11:44:30 | DEBUG    | ALLOWED update_profile | user=Ahmed role=editor
11:44:30 | INFO     | TIMER  update_profile | 0.00 ms
11:44:30 | INFO     | RETURN update_profile | {'status': 'updated', 'user': 'Ahmed', 'data': {'city': 'Lahore', 'password': '***'}}
11:44:30 | INFO     | ---- viewer profile update ----
11:44:30 | INFO     | CALL   update_profile | args=({'name': 'Guest', 'role': 'viewer'}, {'city': 'X'}) kwargs={}
11:44:30 | W